In [1]:
import jax
import jax.numpy as jnp
import optax
from flax import nnx

from eva.mutation import GaussianMutator, apply_mutation

In [2]:
class Linear(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs) -> None:
        self.w = nnx.Param(rngs.params.uniform((dout, din)))
        self.din, self.dout = din, dout

    def __call__(self, x: jax.Array) -> jax.Array:
        return self.w @ x


def fitness(model: nnx.Pytree, model_ref: nnx.Pytree) -> float:
    values = jnp.arange(-1.0, 1.1, 0.1)
    x, y, z = jnp.meshgrid(values, values, values, indexing="ij")
    vectors = jnp.stack([x.ravel(), y.ravel(), z.ravel()], axis=1)
    y_pred = jax.vmap(model)(vectors)
    y_ref = jax.vmap(model_ref)(vectors)
    return -optax.l2_loss(y_pred, y_ref).mean().item()

In [3]:
model_ref = Linear(3, 3, rngs=nnx.Rngs(params=0))
model_ref.w = nnx.Param(
    jnp.array([[1.0, -1.0, 0.0], [0.5, 1.0, 0.5], [-1.0, 1.0, 1.0]])
)
nnx.display(model_ref)

In [4]:
model = Linear(3, 3, rngs=nnx.Rngs(params=0))
curr_fitness = fitness(model, model_ref)
mutator = GaussianMutator(rngs=nnx.Rngs(1), sigma=0.01)
for _ in range(10000):
    child = apply_mutation(model, mutator)
    child_fitness = fitness(child, model_ref)
    if child_fitness > curr_fitness:
        curr_fitness = child_fitness
        model = child

curr_fitness

-5.3187122830422595e-06

In [5]:
nnx.display(model)